In [ ]:
# === TRAINING PRESETS ===
# Dataset: 220,000 samples available

PRESET = 1  # ✓ Fast test - optimized for speed + reward learning

configs = {
    0: {  # Debug - 8k steps (~2 min) - fast iterations to test reward
        'TRAIN_DATA_SIZE': 16_384,
        'N_ENVS': 8,
        'N_STEPS': 256,
        'BATCH_SIZE': 64,
        'NUM_ITERATIONS': 4,
    },
    1: {  # Fast Test - 49k steps (~8 min) - quick feedback
        'TRAIN_DATA_SIZE': 24_576,
        'N_ENVS': 8,
        'N_STEPS': 768,        # Reduced from 1024
        'BATCH_SIZE': 256,     # Keep large for stability
        'NUM_ITERATIONS': 8,   # Reduced from 12
    },
    2: {  # Standard - 131k steps (~20 min)
        'TRAIN_DATA_SIZE': 65_536,
        'N_ENVS': 8,
        'N_STEPS': 1024,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 16,
    },
    3: {  # Extended - 524k steps (~1.5 hours)
        'TRAIN_DATA_SIZE': 131_072,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 16,
    },
    4: {  # Max - 1.04M steps (~3 hours)
        'TRAIN_DATA_SIZE': 220_000,
        'N_ENVS': 8,
        'N_STEPS': 4096,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 32,
    },
}

# Load config
cfg = configs[PRESET]
TRAIN_DATA_SIZE = cfg['TRAIN_DATA_SIZE']
N_ENVS = cfg['N_ENVS']
N_STEPS = cfg['N_STEPS']
NUM_ITERATIONS = cfg['NUM_ITERATIONS']
BATCH_SIZE = cfg['BATCH_SIZE']

# Fixed params - TUNED FOR STABILITY + SPEED
LOOKBACK_WINDOW = 288
HIDDEN_DIM = 256
POLICY_LAYERS = [512, 256, 128]
VALUE_LAYERS = [512, 256]       # Bigger value network to handle high losses
LEARNING_RATE_START = 3e-4      # Stable learning rate
LEARNING_RATE_DECAY = 0.0       # No decay
N_EPOCHS = 10
ENT_COEF = 0.08                 # Balanced exploration
CLIP_RANGE = 0.2                # Smaller policy updates
TARGET_KL  = 0.05               # Prevent early stopping

# Reward normalization bounds (min-max scaling)
REWARD_MIN = -200.0  # Worst case: big loss + penalties
REWARD_MAX = 400.0   # Best case: big win + bonuses

DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_SAVE_PATH = "trading_bot"
VECNORM_SAVE_PATH = "vecnormalize.pkl"

# Calculated
STEPS_PER_ITERATION = N_ENVS * N_STEPS
TOTAL_TIMESTEPS = STEPS_PER_ITERATION * NUM_ITERATIONS

print(f"Preset {PRESET}: {TOTAL_TIMESTEPS:,} steps | {NUM_ITERATIONS} iters | {N_ENVS} envs | {TRAIN_DATA_SIZE:,} samples")
print(f"⚙️ Fast Test Configuration:")
print(f"   - Total steps: {TOTAL_TIMESTEPS:,} (reduced from 98k)")
print(f"   - Iterations: {NUM_ITERATIONS} (reduced from 12)")
print(f"   - Batch size: {BATCH_SIZE} (stable)")
print(f"   - Expected time: ~8-10 minutes")
print(f"   - Reward normalization: [{REWARD_MIN}, {REWARD_MAX}] → [-1, 1]")

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='enable_nested_tensor is True')

from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import torch
import torch.nn as nn
import pandas as pd
import time
from environments.simple_trading_env import SimpleTradingEnv
from environments.trading_enhanced_extractor import TradingEnhancedExtractor
from environments.trading_pattern_memory import TradingPatternMemory

# Load data
df = pd.read_pickle(DATA_PATH)
train_data = df.iloc[0:TRAIN_DATA_SIZE].reset_index(drop=True)
print(f"Loaded {len(df):,} rows | Training on {len(train_data):,} samples")

# Setup policy
policy_kwargs = dict(
    features_extractor_class=TradingEnhancedExtractor,
    features_extractor_kwargs=dict(hidden_dim=HIDDEN_DIM),
    net_arch=dict(pi=POLICY_LAYERS, vf=VALUE_LAYERS),
    activation_fn=torch.nn.GELU,
    ortho_init=False,
)

# Create environments WITH BUILT-IN REWARD NORMALIZATION
vec_env = make_vec_env(
    lambda: Monitor(SimpleTradingEnv(
        train_data, 
        device="cuda", 
        lookback_window=LOOKBACK_WINDOW,
        reward_min=REWARD_MIN,
        reward_max=REWARD_MAX
    )),
    n_envs=N_ENVS
)

# NO VecNormalize needed - rewards normalized in environment!

# Create model
model = PPO(
    "MultiInputPolicy",
    vec_env,
    device="cuda",
    learning_rate=lambda f: LEARNING_RATE_START * (1 - LEARNING_RATE_DECAY * f),
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=CLIP_RANGE,
    ent_coef=ENT_COEF,
    vf_coef=0.5,
    max_grad_norm=0.5,
    target_kl=TARGET_KL,
    stats_window_size=288,
    policy_kwargs=policy_kwargs,
    tensorboard_log="./tensorboard_logs/",
    verbose=2
)

# Reinitialize action network with smaller values for MultiDiscrete action space
# This prevents the model from being too confident in early steps
print("\n🔧 Reinitializing action network with smaller values for MultiDiscrete...")
for module in model.policy.action_net.modules():
    if isinstance(module, nn.Linear):
        nn.init.orthogonal_(module.weight, gain=0.5)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)
print("✓ Action network reinitialized\n")

print(f"📊 Key Changes v24 - MIN-MAX REWARD NORMALIZATION:")
print(f"   ✓ REWARD NORMALIZATION: Built-in min-max scaling")
print(f"      - Range: [{REWARD_MIN}, {REWARD_MAX}] → [-1, 1]")
print(f"      - No VecNormalize needed (faster!)")
print(f"      - Raw rewards preserved for analysis")
print(f"   ✓ REWARD v23: Balance growth + discrete action enforcement")
print(f"      - Redundant action penalty: -50")
print(f"      - Balance change: 0.5x multiplier")
print(f"      - Fast TP (<10 bars): 1.5x bonus")
print(f"      - SL penalty: -15 to -30")
print(f"      - Exploration bonus: +2.0")
print(f"   ✓ TRAINING CONFIG:")
print(f"      - Total steps: {TOTAL_TIMESTEPS:,} (fast test)")
print(f"      - Batch size: {BATCH_SIZE} (stable gradients)")
print(f"      - Learning rate: {LEARNING_RATE_START}")
print(f"      - Value network: {VALUE_LAYERS}")
print(f"   ✓ GOAL: Fast feedback + stable learning\n")

try:
    # Train
    print(f"\nStarting training: {TOTAL_TIMESTEPS:,} steps...")
    model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
except KeyboardInterrupt:
    print("\n⏸ Training interrupted by user")
    
# Save
model.save(MODEL_SAVE_PATH)
print(f"\n✓ Saved: {MODEL_SAVE_PATH}")

# IMPORTANT: Collect pattern memories from all environments
print("\n💾 Collecting pattern memories from all environments...")

# Aggregate pattern memories
aggregated_memory = TradingPatternMemory(capacity=10000)

for i in range(vec_env.num_envs):
    try:
        env = vec_env.envs[i]
        # Unwrap Monitor to get SimpleTradingEnv
        while hasattr(env, 'env'):
            env = env.env
        
        if hasattr(env, 'pattern_memory'):
            # Merge episodes from this environment
            for episode in env.pattern_memory.episodes:
                aggregated_memory.add_episode(episode)
            print(f"  Env {i}: {len(env.pattern_memory.episodes)} episodes")
    except Exception as e:
        print(f"  Env {i}: Error accessing - {e}")

print(f"\n📊 Total episodes collected: {len(aggregated_memory.episodes)}")

if len(aggregated_memory.episodes) > 0:
    # Save aggregated memory
    aggregated_memory.save('training_session_patterns.pkl')

    # Analyze patterns
    patterns = aggregated_memory.get_pattern_distribution()
    print(f"\n📈 Pattern Analysis:")
    print(f"  Episode Win Rate: {patterns['win_rate']:.1%}")

    if 'winning_patterns' in patterns and patterns['winning_patterns']:
        wp = patterns['winning_patterns']
        print(f"\n  ✅ Winning Episodes:")
        print(f"     Avg Trades: {wp['avg_trades']:.1f}")
        print(f"     Avg Length: {wp['avg_length']:.0f} steps")
        print(f"     Avg Return: {wp['avg_return']:.2f}")
        print(f"     Avg Final Balance: ${wp['avg_final_balance']:.2f}")

    if 'losing_patterns' in patterns and patterns['losing_patterns']:
        lp = patterns['losing_patterns']
        print(f"\n  ❌ Losing Episodes:")
        print(f"     Avg Trades: {lp['avg_trades']:.1f}")
        print(f"     Avg Length: {lp['avg_length']:.0f} steps")
        print(f"     Avg Return: {lp['avg_return']:.2f}")
        print(f"     Avg Final Balance: ${lp['avg_final_balance']:.2f}")

    # Export to CSV for detailed analysis
    df_analysis = aggregated_memory.export_to_dataframe()
    df_analysis.to_csv('data/pattern_memory/episode_analysis.csv', index=False)
    print(f"\n✓ Exported episode analysis to CSV")

    # Show top 5 best episodes
    print(f"\n🏆 Top 5 Best Episodes:")
    top_episodes = aggregated_memory.get_top_episodes(n=5, criterion='return')
    for i, ep in enumerate(top_episodes, 1):
        print(f"  {i}. Return: {ep['total_return']:.2f} | "
              f"Balance: ${ep['final_balance']:.2f} | "
              f"Trades: {ep['total_trades']} | "
              f"Win Rate: {ep['win_rate']:.1%}")
else:
    print("\n⚠ No episodes recorded this session")

print("\n✅ Training complete!")